# E2: direct engine versus Hugging Face

E1 proved that a chat client can use the engine over loopback HTTP. E2 asks a different question: **how long does generation take, and when do output token IDs become available?** The user selected direct timing. There is no HTTP server, public networking or deployment in this lesson.

**Status:** prepared, not runtime-validated. E1's separate `cee6f49` run passed 114 tests and is archived in `results/E1_streaming_server.json`. It does not validate these benchmarks.

Use a **fresh Colab T4**. Tests use tiny random models first. The demo briefly holds two full FP32 models for a short parity check, then converts both to FP16 for timing. Do not retain models from D5/E1 in this runtime. Local checks remain static only.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /content/tinyserve
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--index-url",
                "https://pypi.org/simple", "-e", ".[dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)

## 1. Define and test the measurements

**TTFT** is time to first token ID. **Request latency** ends when that request's last token ID is ready. **Inter-token latency (ITL)** is the time between consecutive output IDs for one request. **Total time** additionally includes generation returning to the caller. **Output throughput** is all generated outputs divided by total wall time; do not include prompt tokens in that numerator.

Example: two requests each produce three outputs. Their relative times are `[0.1, 0.3, 0.5]` and `[0.2, 0.4, 0.7]` seconds; total time is 0.8 seconds. TTFT is 100/200 ms, last-ID latency is 500/700 ms, ITLs are `[200, 200]`/`[200, 300]` ms, and throughput is `6 / 0.8 = 7.5` outputs/s.

The tests check that exact arithmetic, single-output behavior, invalid times, prompt skipping in HF's streamer, fixed output counts, alternating trial order, EOS restoration, failure cleanup, routing and actual tiny-model CPU/CUDA integration. D5 engine regressions run in the same subprocess. No weights are downloaded by these tests.

In [ ]:
import gc, json
import torch
from bench.engine_vs_hf import run_case
from bench.hf_baseline import make_prompt_ids
from tinyserve.basics import MODEL_ID, load_model
from tinyserve.envinfo import collect_env, save_result
from tinyserve.models.llama import LlamaForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_d5_engine.py",
                           "tests/test_e2_bench.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()

## 2. First prove a short FP32 continuation

Load the HF TinyLlama weights once and copy them strictly into TinyServe. Both models have the same weights, device and precision. This brief FP32 case uses two 16-token prompts and four outputs each. Exact greedy token equality is required before proceeding; these short FP32 timings are not the performance result.

Every benchmark temporarily disables EOS stopping. An EOS ID can still be selected, but it does not shorten the workload. HF gets a fresh greedy generation configuration and uses DynamicCache. Because HF merges unset fields from its model policy, the benchmark temporarily replaces that default source too, restoring the original object in `finally` without editing its contents. It does not pass the unsupported `use_model_defaults` argument. TinyServe's EOS setting is restored even if a run fails.

Then convert both models to FP16. This avoids keeping extra rounded-weight copies or extra serving engines. Emptying unused allocator cache happens here, not inside timed trials. The selected HF attention implementation will be saved; it need not be the same kernel as our PyTorch gather path.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
our_model = LlamaForCausalLM.from_huggingface(hf_model)
parity_ids = make_prompt_ids(tokenizer, length=16, batch=2, device=DEVICE)
fp32_parity = run_case(our_model, hf_model, parity_ids, new_tokens=4, trials=1,
                      block_size=16, max_batch_tokens=32)
assert fp32_parity["token_ids_equal_in_all_trials"], json.dumps(fp32_parity)
print("FP32 greedy parity passed for both requests.")
our_model.half()
hf_model.half()
gc.collect()
torch.cuda.empty_cache()
print("Timing dtype:", our_model.get_input_embeddings().weight.dtype)
print("HF attention:", hf_model.config._attn_implementation)
print("Driver free/total bytes:", torch.cuda.mem_get_info())

## 3. What is inside each timed run?

All requests arrive together, with equal prompt lengths and no padding. `make_prompt_ids()` reuses the A3 filler-text helper. Batch rows are identical, but each request owns independent KV; no prefix cache exists yet. The tests also exercise different rows.

```text
TinyServe: allocate pool [untimed] -> synchronize -> START -> admit all B requests
           -> step -> synchronize -> output-ID timestamp -> ... -> drain -> END
HF:        synchronize -> START -> generate(use_cache=True, DynamicCache)
           -> skip streamer prompt -> selected IDs -> synchronize -> timestamp
           -> ... -> generate returns -> synchronize -> END
```

CUDA work is asynchronous. Without synchronization a CPU timer can measure work being queued rather than completed. We synchronize on the actual input device. These measurements include Python, observation and synchronization overhead; they are not pure GPU kernel times. HF's callback records IDs only, without tokenizer/text buffering.

Unlike A3, E2 observes TTFT and decode intervals inside the **same full run**, rather than subtracting separate one-token/full runs. Requests updated in one TinyServe step share an observation timestamp. Chunked prefill can delay a later request's first output; that queueing is included in TTFT.

**Important asymmetry:** TinyServe's fixed pool is allocated before timing, like an already-started engine. HF's dynamic cache is allocated within `generate()`. Model loading, prompt construction and token decoding are excluded for both. We record this boundary rather than claim identical allocation work. This is not HTTP latency or an uninstrumented maximum-throughput benchmark.

## 4. Four fixed-work FP16 cases

We vary prompt length `P` over 32 and 128, and simultaneous requests `B` over 1 and 4. Every request produces `G=16` outputs. One full-length warmup runs per mode, followed by three paired trials. Measured order alternates TinyServe/HF, HF/TinyServe, TinyServe/HF to reduce systematic drift bias.

TinyServe allows 128 input rows per forward. Four 128-token prompts therefore need multiple prefill steps. Each request processes `P+G-1` positions; the last sampled output is not fed back. With block size 16, allocate `B * ceil((P+G-1)/16)` blocks, rounding per request. The largest case needs `4 * ceil(143/16) = 36` blocks, or `36 * 360448 = 12976128` FP16 KV bytes (12.375 MiB). All requests fit together; this lesson does not benchmark preemption.

Each run checks exact output counts, no preemption and full pool release. FP16 greedy equality is recorded per request/trial, not demanded universally. If it differs, the record still compares equal fixed output budgets, not identical continuations. No cross-precision tolerance or minimum speedup is invented. A ratio below one is a valid result: our Python gather engine may be slower.

In [ ]:
case_specs = [(1, 32), (4, 32), (1, 128), (4, 128)]
benchmark_cases = []
for batch_size, prompt_length in case_specs:
    case_ids = make_prompt_ids(tokenizer, length=prompt_length, batch=batch_size, device=DEVICE)
    case_result = run_case(our_model, hf_model, case_ids, new_tokens=16, trials=3,
                           block_size=16, max_batch_tokens=128)
    benchmark_cases.append(case_result)
    print("Completed B=%d P=%d; FP16 tokens equal: %s" %
          (batch_size, prompt_length, case_result["token_ids_equal_in_all_trials"]))
    for mode_name in ("tinyserve", "hf"):
        print(mode_name, json.dumps(case_result["summaries"][mode_name], indent=2))
    print("HF/TinyServe total-time ratio:", case_result["hf_over_tinyserve_total_time"])

## 5. Read the ranges, not just the winner

Each summary contains minimum, median, maximum and `n`. Total-time/throughput summaries contain three trial values. TTFT and request-latency summaries pool `B * 3` requests. ITL summaries pool `B * 3 * 15` intervals. Correlated requests and intervals are not independent statistical trials, and three trials do not support a meaningful production p99.

The paired ratio is `HF total time / TinyServe total time`. Above one means TinyServe finished sooner under these exact boundaries. Below one means HF finished sooner. The JSON retains all trials, token IDs and per-request timestamps so a surprising median can be investigated. We do not delete slow trials.

Peak allocated memory includes **both resident models**. TinyServe's baseline also includes its fixed pool, while HF's dynamic cache grows during the measured call. `incremental_peak_allocated_bytes` is peak minus that mode's baseline, not total KV cost or driver-free memory. PyTorch-reserved memory is a different quantity. No per-trial allocator flushing occurs.

The table below gives `min / median / max`. Do not claim a speedup or latency number before running the notebook and reviewing the complete saved payload.

In [ ]:
def range_text(summary):
    return "%.2f / %.2f / %.2f" % (summary["min"], summary["median"], summary["max"])

print("B  P   mode       output tok/s (min/median/max)   TTFT ms (min/median/max)   last-ID ms (min/median/max)")
for case_result in benchmark_cases:
    for mode_name in ("tinyserve", "hf"):
        summary = case_result["summaries"][mode_name]
        print(case_result["batch"], case_result["prompt_tokens"], mode_name,
              range_text(summary["output_tokens_per_s"]), range_text(summary["ttft_ms"]),
              range_text(summary["request_latency_ms"]))

## 6. Save complete evidence, then package the core project

The payload includes test stdout/stderr, the FP32 parity gate, every measured FP16 case/trial, raw token-ready times, exact inputs/outputs, ranges, backend and timing boundaries. `save_result()` adds the source SHA, UTC timestamp, GPU and library versions. Share the **complete JSON**, not only the table or a timing screenshot. E2 stays pending until that output is reviewed.

### Three interview questions
1. Can higher throughput mean worse TTFT? Yes: batching can increase total output rate while later requests wait for prefill.
2. Why observe token times within one full run? It preserves real request delays and avoids subtracting unrelated noisy runs.
3. What makes this comparison fair and what remains different? Same copied weights, dtype, token inputs and budgets; explicit differences in attention backend, cache lifecycle and observer costs.

Guide: `docs/guide/22-engine-vs-hf.md`. Next is **H packaging**: README, blog post, study-guide PDF and CI. After E2 and packaging are done, **pause and go through the entire core project properly again before Parts F/G**.

In [ ]:
assert test_run.returncode == 0 and fp32_parity["token_ids_equal_in_all_trials"]
assert len(benchmark_cases) == 4
assert {(case["batch"], case["prompt_tokens"]) for case in benchmark_cases} == set(case_specs)
assert all(case["dtype"] == "torch.float16" and case["new_tokens"] == 16
           and len(case["trial_records"]) == 3 for case in benchmark_cases)
e2_result = {"test_run": {"returncode": test_run.returncode, "stdout": test_run.stdout,
                          "stderr": test_run.stderr},
             "model_id": MODEL_ID, "scope": "direct; no HTTP, tokenizer or text decoding",
             "weights": "strict HF state_dict copy; both converted from FP32 to FP16 after parity",
             "tf32_allowed": torch.backends.cuda.matmul.allow_tf32,
             "fp32_parity": fp32_parity, "fp16_cases": benchmark_cases,
             "next_checkpoint": "After E2 and H packaging, pause for a full core-project review before stretch work."}
result_path = save_result("E2_engine_vs_hf", e2_result)
print(result_path.read_text())